# Phase7: 学習なしのAI（VLM）との比較 — 「作る」か「借りる」か

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 学習なしで画像を読めるAI（VLM）と、自作のAIを比べ、「作る」か「借りる」かを考える回です。
> - **たとえ話**: 自分で語学を習得するか、通訳さんを雇うか。費用・正確さ・プライバシーで比べる。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-8（VLM）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 30分 ＋ 実行 約30秒

## これまでのおさらい
駐車場のナンバー読み取りシステムを、**自分で作って**（合成画像で学習して）きました。最終評価（test）では、Easy 0.98 / Medium 0.90 / Hard 0.09 でした。

## このNotebookのゴール
**「学習なしで画像を読めるAI」**（VLM）に、**同じナンバー画像**を読ませて、自作のシステムと比べます。

> **問い**: 苦労して**学習データを作り、モデルを学習させ、検査や確信度を整えた**システムは、**学習なしで使える汎用のAI**より、どれだけ良いのか？ **そもそも、作る必要があったのか？**

答えは「どちらが上」という単純なものではなく、**精度・速度・コスト・プライバシー・自信の出し方**など、**複数の面**で違いが出ます。それを、数字で見ます。

## キーワード
| 用語 | 意味 |
|---|---|
| **VLM**（Vision-Language Model） | 画像と言語の両方を扱えるAI。画像を見て、文章で答える |
| **ゼロショット** | そのタスクの**学習データを一切与えずに**、いきなり解かせること |
| **プロンプト** | AIへの質問・指示の文章 |
| **ローカル** | 自分のパソコンの中だけで動かす（画像が外に出ない） |
| **外部サービス** | インターネットの向こうのAI（画像を送る必要がある） |
| **トークン** | AIが扱う文字や画像の単位。多いほど、時間と費用がかかる |
| **自己申告の確信度** | VLMに「どれくらい自信がありますか」と聞いて、答えてもらった数 |
| **95%信頼区間** | 標本が少ないとき、「本当の値は、おそらくこの範囲」という幅。**差がこの幅に収まるなら、誤差の範囲** |

## 🔒 画像の外部送信について（最重要）
このNotebookの比較では、**画像を外部サービスに送りました**。次の点を守っています。

- 送ったのは、**自分で作った合成画像だけ**（実在のナンバーや、実際の車・人の写真は**一切使っていません**）。
- 送る前に、**画像と質問の内容を確認**しました。
- **ローカルのAI**（自分のMacの中だけ）も使い、**外部に出ないほうの結果**と比べられるようにしています。

> ⚠ **実画像（実在のナンバー）では、外部サービスに送ってはいけない場合があります。** ナンバーは**所有者につながる情報**で、取得目的・保存期間・第三者への提供が制限されます。この比較が**合成画像だからできた**ことを、忘れないでください。

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import PIL, numpy, pandas  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from vlm.parse import parse_reply
from vlm.prompts import PROMPTS
from rules.format import is_valid

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
VLM = ROOT / "outputs" / "vlm"
EVAL = ROOT / "data" / "eval" / "test2"
CMP = VLM / "compare_test2_n60.csv"
if not CMP.exists():
    print("比較の結果がありません。VLMへの問い合わせ（src/run_phase7.py ask ...）と集計（report）を実行してください。以降のセルの多くは、結果ができてから動きます。")
else:
    cmp_ = pd.read_csv(CMP, keep_default_na=False)
    cmp_["ok"] = cmp_["pred"] == cmp_["gt"]
    cmp_["conf"] = pd.to_numeric(cmp_["conf"], errors="coerce")
    print("比較の対象:", sorted(cmp_.system.unique()))

上のセルは、道具の読み込みと、比較結果の読み込みです。エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。

## 1. 比べるもの

| システム | 種類 | 画像の行き先 | 学習 |
|---|---|---|---|
| **自作**（検出ResNet18＋CRNN＋書式検査＋確信度） | 自分で学習させた専用システム | 外に出ない | 合成画像で学習（数十万枚） |
| **qwen3.8:27b**（ローカルVLM） | 汎用AI（270億パラメータ） | **外に出ない**（自分のMacで動かす） | なし（ゼロショット） |
| **Claude Sonnet**（VLM） | 汎用AI（大きめ） | **Anthropicに送信**（合成画像のみ） | なし |
| **Claude Haiku**（VLM） | 汎用AI（小さめ・高速） | **Anthropicに送信**（合成画像のみ） | なし |

### 評価データ — 新しい「test2」
最終評価（test）は**使用済み**なので、**一度も使っていない新しいデータ「test2」**（testのシードの1000番目以降。ナンバーの重複なし）を作りました。**難易度ごとに、共通の60枚**（ローカルVLMは100枚も）で、**全システムを同じ画像**で評価します。
- プロンプト・モデル・枚数は、**test2を見る前に固定**しました（動作確認は、別のデータ val で行った）。
- 自作のシステムも、**同じモデル・同じしきい値のまま**（**調整なし**）で読ませています。

### 2通りの質問（プロンプト）
VLMへの質問を、2通り用意しました。**何を教えるか**の違いが、結果にどう響くかを見ます。

In [ ]:
print("【書式つき】（自作のシステムと同じ知識を与える）\n" + "-" * 60)
print(PROMPTS["format"])
print("\n【素の質問】（何も教えない）\n" + "-" * 60)
print(PROMPTS["naive"])

**見るポイント**
- **書式つき**は、「地名は5種類」「かなは10種類」「一連番号の形」を教えています。これは、**自作のシステムが「学習」と「書式ルール」で持っている知識**と、同じものです。**公平に比べるため**、VLMにも同じ知識を与えました。
- **素の質問**は、何も教えず、「ナンバーを読んで」とだけ頼みます。**VLMが元々持つ力だけ**を見ます。
- どちらも「**JSONで、文字列と自信（0〜1）を答えて**」と頼んでいます（自信を答えてもらうのは、あとで確信度として使えるか調べるため）。

### 実際の様子を見る
test2の画像を1枚、VLMが読んだ様子です（Easy と Hard から1枚ずつ）。

In [ ]:
def raw(spec, prompt):
    f = VLM / "raw" / f"test2__{spec}__{prompt}.jsonl"
    return {json.loads(l)["key"]: json.loads(l) for l in open(f, encoding="utf-8")} if f.exists() else {}

R_local, R_sonnet = raw("ollama_qwen3.8_27b-mlx", "format"), raw("claude_sonnet", "format")
labs = {d: [json.loads(l) for l in open(EVAL / d / "labels.jsonl", encoding="utf-8")] for d in ["easy", "medium", "hard"]}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, d in zip(axes, ["easy", "medium", "hard"]):
    lab = labs[d][2]
    ax.imshow(Image.open(EVAL / d / "images" / lab["image"])); ax.axis("off")
    key = f"{d}/{lab['image']}"
    p_l = parse_reply(R_local.get(key, {}).get("reply", "")); p_s = parse_reply(R_sonnet.get(key, {}).get("reply", ""))
    ax.set_title(f"[{d}] 正解: {lab['text']}\nローカルVLM: {p_l['text']}（自信{p_l['confidence']}）\nClaude Sonnet: {p_s['text']}（自信{p_s['confidence']}）", fontsize=8)
fig.tight_layout();

**見るポイント**: Easy はどちらも正解。**Hard は、どちらも読めず**、**自信が低い**（0.1〜0.3程度）と**自己申告**しているものが多い。ここが、あとで重要になります。

## 2. 精度 — 完全一致

各難易度**共通の60枚**（ローカルVLMの素の質問は40枚、Claude Sonnetの素の質問は20枚）。**自作は3シードの平均**です。

| システム | Easy | Medium | Hard | 全体 | 書式OK率 | 解析OK率 |
|---|---|---|---|---|---|---|
| **自作（検出ResNet18＋CRNN）3シード平均** | 0.983 | 0.928 | 0.072 | 0.661 | 0.896 | 1.000 |
| ollama:qwen3.8:27b-mlx [format] | 1.000（60枚） | 0.917（60枚） | 0.000（60枚） | 0.639 | 0.944 | 1.000 |
| ollama:qwen3.8:27b-mlx [naive] | 0.950（40枚） | 0.675（40枚） | 0.000（40枚） | 0.542 | 0.667 | 1.000 |
| claude:haiku [format] | 1.000（60枚） | 0.383（60枚） | 0.000（60枚） | 0.461 | 0.900 | 1.000 |
| claude:sonnet [format] | 1.000（60枚） | 0.950（60枚） | 0.000（60枚） | 0.650 | 0.983 | 1.000 |
| claude:sonnet [naive] | 0.850（20枚） | 0.600（20枚） | 0.000（20枚） | 0.483 | 0.483 | 1.000 |

### 95%信頼区間（標本が少ないので、差が誤差の範囲かを見る）

| システム | Easy | Medium | Hard |
|---|---|---|---|
| 自作（seed0。3シードの代表） | 1.000 [0.94〜1.00]（n=60） | 0.917 [0.82〜0.96]（n=60） | 0.050 [0.02〜0.14]（n=60） |
| ollama:qwen3.8:27b-mlx [format] | 1.000 [0.94〜1.00]（n=60） | 0.917 [0.82〜0.96]（n=60） | 0.000 [-0.00〜0.06]（n=60） |
| ollama:qwen3.8:27b-mlx [naive] | 0.950 [0.83〜0.99]（n=40） | 0.675 [0.52〜0.80]（n=40） | 0.000 [-0.00〜0.09]（n=40） |
| claude:haiku [format] | 1.000 [0.94〜1.00]（n=60） | 0.383 [0.27〜0.51]（n=60） | 0.000 [-0.00〜0.06]（n=60） |
| claude:sonnet [format] | 1.000 [0.94〜1.00]（n=60） | 0.950 [0.86〜0.98]（n=60） | 0.000 [-0.00〜0.06]（n=60） |
| claude:sonnet [naive] | 0.850 [0.64〜0.95]（n=20） | 0.600 [0.39〜0.78]（n=20） | 0.000 [-0.00〜0.16]（n=20） |

In [ ]:
sysorder = ["自作（検出ResNet18＋CRNN）seed0", "ollama:qwen3.8:27b-mlx [format]", "claude:sonnet [format]", "claude:haiku [format]"]
labels = ["自作\n(seed0)", "ローカルVLM\nqwen3.8:27b", "Claude\nSonnet", "Claude\nHaiku"]
fig, ax = plt.subplots(figsize=(10, 3.8))
w = 0.2
for i, d in enumerate(["easy", "medium", "hard"]):
    vals = [cmp_[(cmp_.system == s) & (cmp_.difficulty == d)].ok.mean() for s in sysorder]
    bars = ax.bar(np.arange(4) + (i - 1) * w, vals, w, label=d)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=7)
ax.set_xticks(range(4)); ax.set_xticklabels(labels); ax.set(ylim=(0, 1.12), ylabel="完全一致", title="自作 vs 学習なしのVLM（書式つきの質問・test2・各難易度60枚）"); ax.legend(); ax.grid(axis="y", alpha=0.3)
fig.tight_layout();

### 何が分かるか
1. **Easy は、全員ほぼ満点**（どれも1.00）。きれいな画像なら、**学習なしのVLMでも読める**。
2. **Medium は、自作 0.93、ローカルVLM 0.92、Claude Sonnet 0.95。信頼区間が大きく重なり、区別できません（同等）。** 学習なしのAIが、**専用に学習したシステムに並んだ**。
3. **Claude Haiku（小さめのVLM）は、Medium で 0.38。** 明らかに劣る（信頼区間も重ならない）。**「VLMなら何でも良い」わけではなく、モデルの大きさで大きく違う。**
4. **Hard は、全員ほぼ読めない。** ただし、**ローカルVLMを100枚で評価すると、自作 約0.11（3シード平均。seed0は0.10で信頼区間 0.06〜0.17）、VLM 0.00（信頼区間 0.00〜0.04）で、自作が有意に上**です。**劣化した画像で学習した**自作のほうが、わずかに読める。

ローカルVLMを、各難易度**100枚**で評価した結果（標本を増やして、区間を狭めた）:

| システム | Easy | Medium | Hard | 全体 | 書式OK率 | 解析OK率 |
|---|---|---|---|---|---|---|
| **自作（検出ResNet18＋CRNN）3シード平均** | 0.980 | 0.910 | 0.113 | 0.668 | 0.890 | 1.000 |
| ollama:qwen3.8:27b-mlx [format] | 1.000（100枚） | 0.900（100枚） | 0.000（100枚） | 0.633 | 0.940 | 1.000 |

| システム | Easy | Medium | Hard |
|---|---|---|---|
| 自作（seed0。3シードの代表） | 1.000 [0.96〜1.00]（n=100） | 0.920 [0.85〜0.96]（n=100） | 0.100 [0.06〜0.17]（n=100） |
| ollama:qwen3.8:27b-mlx [format] | 1.000 [0.96〜1.00]（n=100） | 0.900 [0.83〜0.94]（n=100） | 0.000 [0.00〜0.04]（n=100） |

> 💡 **学習データが要らない**のは大きな利点です。自作は、**合成データ生成器・検出器・OCRの学習・検査・しきい値決定**に、**非常に多くの手間**をかけました。それでも、**きれいな〜やや劣化した条件**では、**学習なしのVLMが追いついてしまう**。この事実は、率直に受け止めるべきです。

## 3. 質問の仕方で、結果は大きく変わる

**書式を教えるか、教えないか**で、VLMの結果はどう変わるでしょう。**同じ画像**で比べます（素の質問に使った画像に、書式つきの結果を絞って比較）。

| モデル | 画像 | 素の質問 | 書式つき |
|---|---|---|---|
| ollama:qwen3.8:27b-mlx | 120枚 | 0.542（書式OK 0.667） | 0.633（書式OK 0.942） |
| claude:sonnet | 60枚 | 0.483（書式OK 0.483） | 0.650（書式OK 0.983） |

**見るポイント**
- **書式を教えると、完全一致が大きく上がります**（ローカル 0.54→0.63、Claude Sonnet 0.48→0.65）。**書式に合う文字列（書式OK）の割合も、0.5〜0.7 → 0.94〜0.98 に**。
- **素の質問**では、**書式に合わない答え**（地名を誤る、桁数が違う、ハイフンの位置が違う）が多い。VLMは、「ナンバー」の一般的な知識はあっても、**この課題の書式**は知らない。
- → **VLMでも、「課題の知識を教える」ことが重要**。これは、自作のシステムが**書式ルール（Phase4）と学習データ（Phase0〜3）で、課題の知識を組み込んだ**のと、本質的に同じです。**プロンプトを工夫する**＝**知識をAIに渡す作業**。

## 4. 自信の使い方 — VLMは「分からない」と言える

Phase3 で、自作のAIは**「分からない」と言えない**（板が無くても答える、確信度は粗い手がかり）と分かりました。VLMは、どうでしょう。**VLM自身に「自信は？」と聞いた答え**（自己申告）を、難易度別に見ます。**しきい値は0.9で固定**（調整していません）。

| システム | 難易度 | 平均の自信 | 自動処理（自信が高い）の割合 | その中の誤り率 |
|---|---|---|---|---|
| ollama:qwen3.8:27b-mlx [format] | easy | 0.98 | 0.983 | 0.000 |
| ollama:qwen3.8:27b-mlx [format] | medium | 0.83 | 0.467 | 0.000 |
| ollama:qwen3.8:27b-mlx [format] | hard | 0.22 | 0.000 | nan |
| ollama:qwen3.8:27b-mlx [naive] | easy | 0.96 | 0.925 | 0.000 |
| ollama:qwen3.8:27b-mlx [naive] | medium | 0.90 | 0.550 | 0.000 |
| ollama:qwen3.8:27b-mlx [naive] | hard | 0.34 | 0.000 | nan |
| claude:haiku [format] | easy | 0.96 | 1.000 | 0.000 |
| claude:haiku [format] | medium | 0.79 | 0.300 | 0.111 |
| claude:haiku [format] | hard | 0.34 | 0.000 | nan |
| claude:sonnet [format] | easy | 0.94 | 0.967 | 0.000 |
| claude:sonnet [format] | medium | 0.82 | 0.467 | 0.000 |
| claude:sonnet [format] | hard | 0.10 | 0.000 | nan |
| claude:sonnet [naive] | easy | 0.92 | 0.750 | 0.000 |
| claude:sonnet [naive] | medium | 0.72 | 0.150 | 0.000 |
| claude:sonnet [naive] | hard | 0.10 | 0.000 | nan |
| 自作（3シード込み・seq確信度） | easy | 0.74 | 0.911 | 0.006 |
| 自作（3シード込み・seq確信度） | medium | 0.65 | 0.806 | 0.034 |
| 自作（3シード込み・seq確信度） | hard | 0.05 | 0.022 | 0.000 |

In [ ]:
x = cmp_[cmp_.system.str.contains("format") & cmp_.conf.notna()]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, d in zip(axes, ["easy", "medium", "hard"]):
    s = x[(x.difficulty == d) & (x.system == "claude:sonnet [format]")]
    bins = np.linspace(0, 1, 11)
    ax.hist(s[s.ok].conf, bins=bins, alpha=0.7, label="正解", color="C2"); ax.hist(s[~s.ok].conf, bins=bins, alpha=0.8, label="誤り", color="C3")
    ax.set(title=f"Claude Sonnet・{d}: 自己申告の自信", xlabel="自信（0〜1）"); ax.legend(fontsize=8)
fig.tight_layout();

**見るポイント**
- **Easy**: 自信は**0.9以上**に集中し、ほぼ全部が**正解**。
- **Medium**: 自信が**0.5〜0.9くらいに散らばり**、**0.9以上のものは、正解のみ**（赤＝誤りは、自信が低い側に）。
- **Hard**: 自信は**0.1〜0.3**。**「読めない」と、自分で分かっている**（自信0.9以上は、ゼロ）。誤りは、ほぼ全部が**低い自信**の側。
- 自作の確信度（seq）と比べると、**VLMのほうが「分からない」を、自然に言える**。AUROC（正解と誤りを分ける力）は、VLM 0.985〜0.999、自作 0.96〜0.98で、**VLMの自己申告のほうが、やや高い**。

> ⚠ **注意（慎重に読む）**: ①標本が小さい（Medium で自信0.9以上が約28枚。**誤り0件でも、本当の誤り率は最大で約12%**の可能性）。②**自己申告の自信が、本当に当たるか（較正）は、十分に確かめていない**。③VLMは、**ある難易度では保守的すぎる**（Medium で、自動処理に回せるのは約47%。自作は約81%）。**安全だが、人手が増える**。

## 5. 速度・コスト・プライバシー

| システム | 1枚あたりの時間(秒) | 入力トークン/枚 | 出力トークン/枚 | 1,000枚あたりの金額（報告値） | 画像の外部送信 |
|---|---|---|---|---|---|
| **自作（3シード分まとめて）** | 63ミリ秒 | — | — | 0円（自分のMacで動く） | **なし** |
| ollama:qwen3.8:27b-mlx [format] | 3.9 | 382 | 30 | 0ドル相当 | なし（ローカル） |
| ollama:qwen3.8:27b-mlx [naive] | 2.4 | 167 | 24 | 0ドル相当 | なし（ローカル） |
| claude:haiku [format] | 17.7 | 23,878 | 1,961 | 24ドル相当 | **あり（Anthropicへ）** |
| claude:sonnet [format] | 11.4 | 29,648 | 526 | 32ドル相当 | **あり（Anthropicへ）** |
| claude:sonnet [naive] | 8.1 | 29,579 | 305 | 29ドル相当 | **あり（Anthropicへ）** |

**見るポイント**
- **速度**: 自作は**1枚あたり約0.06秒**（検出・古典・CRNN3つ分まとめて）。**ローカルVLMは約4秒（約60倍）、Claude Sonnetは約11秒（約180倍）**。ゲートで車が待つ駐車場には、**VLMは遅すぎる**。
- **コスト**: 自作は**0円**（自分のMacで、推論は一瞬）。**Claude は、1,000枚あたり約24〜32ドル相当**（サブスク経由の報告値）。⚠ **今回はClaude Code CLI経由で呼んだため、入力トークンが1枚あたり約2.4〜3万と、非常に多い**（CLIの道具・設定の説明文が毎回付く）。**API を直接使えば、画像の分だけ（数百トークン）で済み、もっと安い**はずです。**この金額は、上限の目安**と読んでください。
- **プライバシー**: **自作とローカルVLMは、画像が外に出ない。** Claude は**Anthropicに送信**される。**実際のナンバーでは、これが大きな制約**。

## 6. 併用する — 互いの弱点を補う（探索的）

自作は**「分からない」の出し方が粗く**、VLMは**自信を言葉で言える**。**併用**できないでしょうか。

**方式**: ①自作が「自動処理」と判断したら、**そのまま採用**。②そうでないとき、**VLMの自信が0.9以上**（かつ書式OK）なら、**VLMの答えを採用**。③それでも決まらなければ、**人へ**。

| 方式 | 難易度の混ぜ方 | 自動処理率 | 自動処理の誤り率 |
|---|---|---|---|
| 自作だけ | 均等 | 0.580 | 0.0193 |
| 自作＋claude:sonnet | 均等 | 0.635 | 0.0175 |
| 自作だけ | 昼間中心 | 0.791 | 0.0149 |
| 自作＋claude:sonnet | 昼間中心 | 0.867 | 0.0135 |
| 自作だけ | 均等 | 0.580 | 0.0193 |
| 自作＋ollama:qwen3.8:27b-mlx | 均等 | 0.631 | 0.0176 |
| 自作だけ | 昼間中心 | 0.791 | 0.0149 |
| 自作＋ollama:qwen3.8:27b-mlx | 昼間中心 | 0.864 | 0.0135 |

**見るポイント**
- **昼間中心で、自動処理率が 0.79 → 0.87**（約8ポイント増）、**自動処理の誤り率は、むしろ少し下がる**（1.49% → 1.35%）。**人手が減り、誤りは増えない。**
- VLMは、**自作が「要確認」にしたもの**のうち、**自信を持って読めるもの**を拾ってくれる。**VLMを全部の画像に使う**（遅く・高い）のではなく、**自作が迷ったものだけ**に使えば、**速度とコストの問題も小さくなる**（自作が自動処理に回さないのは、昼間中心で約2割、均等で約4割）。
- ⚠ **探索的な分析**です（test2を見ながら考えた方式なので、**参考値**）。標本も小さい。本格的に使うなら、**別のデータで、しきい値も含めて検証し直す**必要があります。

## 7. 結局、どう使い分けるか

| 観点 | 自作（学習した専用システム） | 汎用VLM（学習なし） |
|---|---|---|
| **精度（Easy・Medium）** | 高い | **同等**（大きめのモデルなら）。小さいモデルは劣る |
| **精度（Hard）** | わずかに読める（約10%） | **ほぼ読めない** |
| **開発の手間** | **非常に大きい**（データ・学習・検査・評価） | **小さい**（質問を書くだけ） |
| **速度** | **非常に速い**（約0.06秒） | 遅い（4〜11秒） |
| **コスト（運用）** | ほぼゼロ | 外部は高い。ローカルは電気代のみ |
| **プライバシー** | 画像が外に出ない | 外部は**送信が必要**。ローカルは外に出ない |
| **「分からない」の出し方** | 粗い（確信度は手がかり程度） | **自然に言える**（自己申告。較正は未検証） |
| **条件が変わったとき** | 学習し直し（分布シフトに弱い） | **柔軟**（質問を変えるだけ。ただし書式の知識は教える必要あり） |
| **誤りの傾向** | 形の似た文字の取り違え | もっともらしい別の文字列を答える |

### 使い分けの目安
- **まず試すなら VLM**。**数時間でベースライン**ができる（自作に数日〜数週間かけた前に、**どこまで行けるか**が分かる）。**学習データのラベル付けの補助**にも使える。
- **速度・コスト・プライバシーが重要な本番**（ゲートの出入口、大量処理）は、**自作（または専用の軽いモデル）**。
- **併用**: 専用システムが**迷ったものだけ**、VLMに聞く。**人手**を減らせる可能性がある。
- **Hard のように、画像に情報が無い場合は、どちらも無力**。**撮影条件の改善**（Phase5・6の結論）が、最優先。

## 🧪 試してみよう

### 演習1: 質問を自分で書いて、ローカルVLMに読ませる
**やること**: 下のセルの **`my_prompt`** を書き換えて実行します。**ローカルのAI（Ollama）**が動いていれば、test2の画像を1枚読みます（**画像は外に出ません**）。たとえば、**書式の一部だけ**教える（地名の一覧だけ、など）と、結果はどう変わりますか？
- ⚠ Ollamaが動いていない場合は、メッセージが出るだけです（エラーにはなりません）。
- 🔒 **ローカル以外には送りません**（`OllamaClient` は、127.0.0.1以外を拒否します）。

In [ ]:
my_prompt = PROMPTS["naive"]       # ← 書き換えてみる。例: PROMPTS["format"] や、自分で書いた質問
try:
    from vlm.clients import OllamaClient
    client = OllamaClient("qwen3.8:27b-mlx")
    lab = labs["medium"][5]
    reply, meta = client.generate(my_prompt, EVAL / "medium" / "images" / lab["image"])
    print("正解:", lab["text"]); print("返事:", reply.strip()[:300]); print("→ 解析:", parse_reply(reply), f"（{meta['wall_s']:.1f}秒）")
except Exception as e:
    print("ローカルVLMに接続できませんでした（Ollamaが動いているか確認してください）:", str(e)[:120])

### 演習2: VLMの自信のしきい値を変える
**やること**: 下のセルの **`th = 0.9`** を **`0.8`**、**`0.95`**、**`0.99`** に書き換えて実行します。
- **見るポイント**: しきい値を**下げる**と、**自動処理できる割合**と**その中の誤り率**は、どう変わりますか？ Claude Haiku は、Sonnetと違いますか？
- **分かること**: **VLMの自己申告の自信を、しきい値で使う**ときの、トレードオフ（Phase4と同じ考え方）。

In [ ]:
th = 0.9       # ← 0.8 / 0.95 / 0.99 に書き換えて実行
rows = []
for s in ["ollama:qwen3.8:27b-mlx [format]", "claude:sonnet [format]", "claude:haiku [format]"]:
    x = cmp_[(cmp_.system == s) & cmp_.conf.notna()]
    x = x[[is_valid(p) for p in x.pred]]
    hi = x[x.conf >= th]
    rows.append({"システム": s, "しきい値": th, "自動処理の割合": round(len(hi) / len(cmp_[cmp_.system == s]), 3), "その中の誤り率": round((~hi.ok).mean(), 4) if len(hi) else None, "自動処理した枚数": len(hi)})
pd.DataFrame(rows).set_index("システム")

### 演習3: Hard で、VLMは何を答えているか
**やること**: 下のセルを実行し、Claude Sonnet が Hard の画像に**何を答えたか**を見ます。
- **見るポイント**: 答えは**書式に合っていますか**？ **自信**は？ 正解と**どれくらい違い**ますか？
- **分かること**: VLMは、読めないときも、**もっともらしい答え**を出す。**自信が低い**ことで、読めていないと分かる。自作の「読み取り不可」（書式が直せない）と、**どちらが親切か**。

In [ ]:
h = cmp_[(cmp_.system == "claude:sonnet [format]") & (cmp_.difficulty == "hard")].head(8)
h[["gt", "pred", "conf"]].rename(columns={"gt": "正解", "pred": "VLMの答え", "conf": "自己申告の自信"}).reset_index(drop=True)

### 考えてみよう（コード不要）
1. **自作に費やした手間**は、**VLMで代替できた**のでしょうか？ 何が代替でき、**何が代替できなかった**でしょう？（ヒント: Hard、速度、プライバシー、確信度の較正）
2. **VLMを本番に使う**としたら、**どんな条件**が必要ですか？ **実在のナンバー**を扱う場合は？
3. 「**書式を教えると良くなる**」ことは、**他のAI応用**（たとえば、天気図を解説させる、災害の写真を判定させる）で、**どう活かせ**ますか？
4. **併用**の考え方を、**他の場面**（たとえば、専用AIが迷ったら、人に回す前に別のAIに聞く）に応用できますか？ **どんなリスク**がありますか？

---

## ✅ まとめ

- **学習なしのVLMは、書式を教えれば、Easy・Mediumで、専用に学習した自作システムに並んだ**（大きめのモデルなら）。**小さなモデル（Haiku）は、Mediumで大きく劣る。**
- **Hardは、どちらも読めない**（自作がわずかに上。有意）。**画像に情報が無い問題は、AIの種類では解決しない。**
- **質問（プロンプト）で、課題の知識を教える**ことが重要。**自作の学習・書式ルールと、本質的に同じ**。
- **VLMは「分からない」と言える**（自己申告の自信）。Hardでは自信が低い。**自作の確信度より、扱いやすいかもしれない**（較正は未検証）。
- **速度・コスト・プライバシー**では、**自作が圧倒的**（約0.06秒、ゼロ円、画像が外に出ない。VLMは約4〜11秒、外部は高く、画像を送る）。
- **併用**（自作が迷ったものだけVLMに聞く）で、**人手を減らせる可能性**がある（探索的）。
- **作る価値**は、**精度だけでなく、速度・コスト・プライバシー・制御**にある。**まず VLM で試し、本番に必要な条件で、作るか借りるか決める**のが良い。

🎉 **全フェーズ（Phase0〜7）完了。** 最終レポートは [REPORT.md](../REPORT.md) です。